<a href="https://colab.research.google.com/github/GowthamPitla/smart-audio-text-accuracy/blob/main/SHL_intern_assign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Upload the ZIP File

In [2]:
from google.colab import files
uploaded = files.upload()  # Upload 'shl-intern-hiring-assessment.zip'

KeyboardInterrupt: 

### Step 2: Extract the ZIP File

In [1]:
import zipfile

zip_path = "/content/shl-intern-hiring-assessment.zip"

if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("✅ ZIP extracted successfully.")
else:
    print("❌ The uploaded file is not a valid ZIP. Please re-upload.")

❌ The uploaded file is not a valid ZIP. Please re-upload.


### Step 3: Install Required Packages

In [2]:
!pip install SpeechRecognition
!pip install language_tool_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 24.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 kB 4.7 MB/s eta 0:00:00


### Step 4: Import Libraries

In [3]:
import os
import speech_recognition as sr
import pandas as pd
import language_tool_python
from difflib import SequenceMatcher

### Step 5: Set Paths

In [4]:
audio_folder = "/content/shl-intern-hiring-assessment/Dataset/audio_train"
train_csv_path = "/content/shl-intern-hiring-assessment/Dataset/train.csv"

### Step 6: Convert Audio to Text

In [5]:
r = sr.Recognizer()
audio_to_text = {}

for filename in sorted(os.listdir(audio_folder)):
    if filename.endswith(".wav") or filename.endswith(".mp3"):
        file_path = os.path.join(audio_folder, filename)
        with sr.AudioFile(file_path) as source:
            try:
                audio = r.record(source)
                text = r.recognize_google(audio)
                audio_to_text[filename] = text
            except sr.UnknownValueError:
                audio_to_text[filename] = "[Unrecognized Speech]"
            except Exception as e:
                audio_to_text[filename] = f"[Error: {e}]"

FileNotFoundError: [Errno 2] No such file or directory: '/content/shl-intern-hiring-assessment/Dataset/audio_train'

### Step 7: Load CSV and Map Predictions

In [ ]:
df = pd.read_csv(r"/Users/harshithatelugu/Desktop/dataset")
df["predicted_text"] = df["filename"].map(audio_to_text)

### Step 8: Calculate Similarity Score

In [ ]:
def text_similarity(a, b):
    return round(SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio() * 100, 2)

df["similarity_score"] = df.apply(lambda row: text_similarity(row["transcript"], row["predicted_text"]), axis=1)

### Step 9: Calculate Grammar Score

In [ ]:
tool = language_tool_python.LanguageTool('en-US')

def grammar_score(text):
    matches = tool.check(text)
    num_errors = len(matches)
    words = len(text.split())
    score = 100 if words == 0 else max(0, round((1 - num_errors / words) * 100, 2))
    return score

df["grammar_score"] = df["predicted_text"].apply(grammar_score)
df[["filename", "transcript", "predicted_text", "similarity_score", "grammar_score"]].head()